# Neural Oral Bible Translation: Whisper-Small + LoRA on BibleTTS Yoruba

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paviomuniz/deepwork/blob/main/notebooks/01_whisper_lora_training.ipynb)

This notebook executes **Phase 2 & 3** of the project specification on a Google Colab Tesla T4 (16 GB VRAM):
1. **Configuration A (Zero-Shot Baseline):** Vanilla `openai/whisper-small` evaluated on the held-out Gospel of John (`JHN`).
2. **Configuration B (LoRA without SpecAugment):** Parameter-efficient fine-tuning (`r=32`, `alpha=64`) on raw log-mel features.
3. **Configuration C (LoRA + SpecAugment):** Full fine-tuning with frequency ($F=27$) and time ($T=100$) masking.
4. **Evaluation & Audio Inspection:** WER, CER (diacritic-preserving & diacritic-stripped), T-NEER entity audit, and side-by-side audio playback.

In [ ]:
# 1. Check GPU & Clone Repository
!nvidia-smi
!git clone https://github.com/paviomuniz/deepwork.git || (cd deepwork && git pull)
%cd deepwork
!pip install -q -r requirements.txt
!pip uninstall -y torchao

In [ ]:
# 2. Imports & Reproducibility
import torch
from IPython.display import Audio, display
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

from src.config import ProjectConfig, set_seed
from src.dataset import (
    DataCollatorSpeechSeq2SeqWithPadding,
    filter_and_prepare_split,
    load_and_split_bibletts,
)
from src.evaluate import (
    compute_asr_metrics,
    compute_theological_entity_audit,
    make_compute_metrics,
)
from src.model import build_processor, build_whisper_lora_model

config = ProjectConfig()
set_seed(config.seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
# 3. Load & Preprocess BibleTTS Yoruba (Canonical Book-Level Split)
processor = build_processor(config)
raw_splits = load_and_split_bibletts(config, nt_only=True)
print(raw_splits)

train_ds = filter_and_prepare_split(raw_splits['train'], processor, config, num_proc=2)
val_ds = filter_and_prepare_split(raw_splits['validation'], processor, config, num_proc=2)
test_ds = filter_and_prepare_split(raw_splits['test'], processor, config, num_proc=2)

In [ ]:
# 4. Configuration A: Zero-Shot Baseline on Gospel of John (JHN)
baseline_model = build_whisper_lora_model(config, use_lora=False).to(device)
eval_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=baseline_model.config.decoder_start_token_id,
    apply_spec_aug=False,
)

baseline_args = Seq2SeqTrainingArguments(
    output_dir='checkpoints/baseline',
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    report_to=['none'],
)

baseline_trainer = Seq2SeqTrainer(
    args=baseline_args,
    model=baseline_model,
    data_collator=eval_collator,
    compute_metrics=make_compute_metrics(processor.tokenizer),
    processing_class=processor.feature_extractor,
)

baseline_metrics = baseline_trainer.evaluate(eval_dataset=test_ds, metric_key_prefix='test')
print('Configuration A (Zero-Shot Baseline) Metrics:', baseline_metrics)
del baseline_model
torch.cuda.empty_cache()

In [ ]:
# 5. Configuration B (LoRA without SpecAugment) & Configuration C (LoRA + SpecAugment)
from src.train import run_training

# Run Configuration B
metrics_b = run_training(config, use_spec_augment=False)

# Run Configuration C (Recommended)
metrics_c = run_training(config, use_spec_augment=True)

In [ ]:
# 6. Qualitative Inspection & Audio Playback on Gospel of John
sample = raw_splits['test'][0]
print('Verse ID:', sample['verse_id'])
print('Ground Truth:', sample['text'])
display(Audio(sample['audio']['array'], rate=sample['audio']['sampling_rate']))